<a href="https://colab.research.google.com/github/catorrecampo-sys/-GE-120-1A-Final-Project-Group-5/blob/main/Leveling_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [100]:
import math

# CHECK / IMPORT MATPLOTLIB
try:
    import matplotlib.pyplot as plt
    print('Matplotlib imported successfully.')
except ImportError:
    print('Error! Matplotlib is not installed.')
    print('Install using: pip install matplotlib')
    exit()

# CHECK / IMPORT PANDAS
try:
    import pandas as pd
    print('Pandas imported successfully.')
except ImportError:
    print('Error! Pandas is not installed.')
    print('Install using: pip install pandas') # for context, I did not use force %pip install pandas since we're not sure if they're using jupyter
    exit()

Pandas imported successfully.


Objective 1

In [101]:
class FileHandler:
    '''
    Handles:
        - file validation
        - csv reading
        - leveling type detection
    '''

    def __init__(self):
        self.extension = '.csv'
        self.three_wire_columns = [
            'BS_point',
            'BS_upper',
            'BS_middle',
            'BS_lower',
            'FS_point',
            'FS_upper',
            'FS_middle',
            'FS_lower'
        ]
        self.differential_columns = [
            'BS_point',
            'BS_Reading',
            'BS_Dist',
            'FS_point',
            'FS_Readings',
            'FS_Dist'
        ]
        self.filename = ''
        self.leveling_data = None
        self.leveling_type = ''

    def read_file(self):
        '''
        '''

        while True:
            self.filename = input('Enter name of csv file to open: ')

            # Automatically add .csv
            if not self.filename.endswith(self.extension):
                self.filename += self.extension

            try:
                self.leveling_data = pd.read_csv(self.filename)

            except FileNotFoundError:
                print('Error! File not found.')
                continue

            except pd.errors.EmptyDataError:
                print('Error! CSV file is empty.')
                continue

            except pd.errors.ParserError:
                print('Error! Invalid CSV formatting.')
                continue

            # Detect leveling type
            self.detect_leveling_type()

            # If valid format
            if self.leveling_type != '':
                break

        return self.leveling_data, self.leveling_type

    def detect_leveling_type(self):
        '''
        '''

        columns = self.leveling_data.columns.tolist()

        # 3-WIRE
        if all(col in columns for col in self.three_wire_columns):
            self.leveling_type = '3-Wire'

            print(f'\nOpening {self.filename}')
            print('Detected: 3-Wire Leveling Data')

        # DIFFERENTIAL
        elif all(col in columns for col in self.differential_columns):
            self.leveling_type = 'Differential'

            print(f'\nOpening {self.filename}')
            print('Detected: Differential Leveling Data')

        # INVALID
        else:
            self.leveling_type = ''

            print('Error! CSV format is not recognized.')
            print('Check column names and file structure.')

In [102]:
# Create FileHandler object to manage CSV input and detection
handler = FileHandler()

# Read CSV file and get dataset + leveling type
data, level_type = handler.read_file()

Enter name of csv file to open: leveling_data

Opening leveling_data.csv
Detected: Differential Leveling Data


In [103]:
data.head()

,BS_point,BS_Reading,BS_Dist,FS_point,FS_Readings,FS_Dist
0,BM1,0.70,38,TP 1,1.990,38
1,TP 1,1.53,26,TP 2,0.930,26
2,TP 2,1.57,15,TP 3,1.000,11
3,TP 3,1.66,16,TP 4,1.020,16
4,TP 4,1.82,18,TP 5,0.836,19


Objective 2-4

In [104]:
class LevelLoop:
    '''
    '''

    def __init__(self):
        self.three_wire = '3-Wire'
        self.differential = 'Differential'
        self.HI_list = []
        self.elev_list = []

    def compute_elev(self):
        '''
        '''

        # Fixed: Wrapped input() in float() to allow math operations later
        self.bm_elev = float(input('Enter BM Elevation: '))
        self.current_elev = self.bm_elev

        if level_type == self.three_wire:
            for idx, row in data.iterrows():
                # average BS wire readings using built-in round()
                data.loc[idx, 'BS_Reading'] = round((
                    row['BS_upper'] +
                    row['BS_middle'] +
                    row['BS_lower']
                ) / 3, 3)

                # average FS wire readings using built-in round()
                data.loc[idx, 'FS_Readings'] = round((
                    row['FS_upper'] +
                    row['FS_middle'] +
                    row['FS_lower']
                ) / 3, 3)
        else:
            pass

        for idx, row in data.iterrows():
            # compute HI
            HI_value = self.current_elev + data.loc[idx, 'BS_Reading']
            self.HI_list.append(HI_value)

            # compute new elevation
            stat_elev = HI_value - data.loc[idx, 'FS_Readings']
            self.elev_list.append(stat_elev)

            self.current_elev = stat_elev

        data['Elevation'] = self.elev_list

        return data, self.HI_list, self.elev_list

    def misclosure(self):
        if level_type == self.three_wire:
            running_dist = 0
            for idx, row in data.iterrows():
                # average BS wire readings
                data.loc[idx, 'BS_Dist'] = (
                    row['BS_upper'] -
                    row['BS_lower']
                ) * 100

                # average FS wire readings
                data.loc[idx, 'FS_Dist'] = (
                    row['FS_upper'] -
                    row['FS_lower']
                ) * 100

                data.loc[idx, 'Total_Dist'] = data.loc[idx, 'BS_Dist'] + data.loc[idx, 'FS_Dist']

                running_dist += data.loc[idx, 'Total_Dist']
                data.loc[idx, 'Dist_from_BM'] = running_dist
        else:
            running_dist = 0
            for idx, row in data.iterrows():
                data.loc[idx, 'Total_Dist'] = row['BS_Dist'] + row['FS_Dist']

                running_dist += data.loc[idx, 'Total_Dist']
                data.loc[idx, 'Dist_from_BM'] = running_dist

        tot_dist = data['Total_Dist'].sum()
        tot_bs = data['BS_Reading'].sum()
        tot_fs = data['FS_Readings'].sum()
        misclosure = round((tot_bs - tot_fs), 3)

        try:

            # convert meters to kilometers
            tot_dist_km = tot_dist / 1000

            allowable_first = 0.004 * (math.sqrt(tot_dist_km))
            allowable_second = 0.008 * (math.sqrt(tot_dist_km))
            allowable_third = 0.012 * (math.sqrt(tot_dist_km))

            abs_error = abs(misclosure)

            if abs_error <= allowable_first:
                order = 'First Order Accuracy'

            elif abs_error <= allowable_second:
                order = 'Second Order Accuracy'

            elif abs_error <= allowable_third:
                order = 'Third Order Accuracy'

            else:
                order = 'Rejected / Low Accuracy'

            return round(misclosure, 4), order

        except Exception as e:
            print(f'Error evaluating accuracy: {e}')
            return misclosure, 'Unknown'

    def adjusted_elev(self, misclosure):
        for idx, row in data.iterrows():
            data.loc[idx, 'Adj_Elev'] = (misclosure*-1)*row['Dist_from_BM'] + row['Elevation']

In [105]:
# Create FileHandler object to manage CSV input and detection
solver = LevelLoop()

# Read CSV file and get dataset + leveling type
data_2, HI_list, elev_list = solver.compute_elev()

data_2.head()

Enter BM Elevation: 100


,BS_point,BS_Reading,BS_Dist,FS_point,FS_Readings,FS_Dist,Elevation
0,BM1,0.70,38,TP 1,1.990,38,98.710
1,TP 1,1.53,26,TP 2,0.930,26,99.310
2,TP 2,1.57,15,TP 3,1.000,11,99.880
3,TP 3,1.66,16,TP 4,1.020,16,100.520
4,TP 4,1.82,18,TP 5,0.836,19,101.504


Objective 5-6

In [106]:
class MakeReport:
    def __init__(self, final_dataframe, error_val, acc_order):
        self.df = final_dataframe
        self.error = error_val
        self.order = acc_order

    def save_graph(self, base_name):
        # Graph filename using the base name
        graph_filename = base_name + '_profile.png'

        try:
            plt.figure(figsize=(10, 5))
            x_vals = self.df['Dist_from_BM']
            y_vals = self.df['Adj_Elev']

            # Plot the main line
            plt.plot(x_vals, y_vals, marker='o', linestyle='-', color='darkred', label='Adjusted Elevation')

            # Area under the line
            plt.fill_between(x_vals, y_vals, color='red', alpha=0.3)

            # Combine the user's input name with "Elevation Profile"
            plt.title(f'{base_name} Elevation Profile')
            plt.xlabel('Distance from BM (m)')
            plt.ylabel('Elevation (m)')

            # Grid lines transparency
            plt.grid(True, alpha=0.5)
            plt.legend()

            plt.savefig(graph_filename, bbox_inches='tight')
            plt.close()

            print(f'Graph automatically saved as {graph_filename}')

        except Exception as e:
            print('Oops, error saving graph image:', e)

    def save_to_txt(self):
        raw_name = input('Type the base project name (e.g. Site_A): ')

        # In case they accidentally typed ".txt"
        base_name = raw_name.replace('.txt', '').replace('.html', '')

        # Create the new filename with _report attached
        filename = base_name + '_report.txt'

        try:
            text_file = open(filename, 'w')
            text_file.write("--- LEVELING COMPUTATION REPORT ---\n\n")
            text_file.write(self.df.to_string(index=False))
            text_file.write("\n\n--- MISCLOSURE AND ACCURACY SUMMARY ---\n")
            text_file.write(f"Error of Misclosure: {self.error}\n")
            text_file.write(f"Order of Accuracy: {self.order}\n")
            text_file.close()

            print('\nOkay, report saved as', filename)

            # Saving Graph
            self.save_graph(base_name)

        except Exception as e:
            print('Oops, error saving text file:', e)

    def save_to_html(self):
        raw_name = input('Type the base project name (e.g. Site_A): ')

        # In case they accidentally typed ".html"
        clean_name = raw_name.replace('.txt', '').replace('.html', '')

        # Create the new filename with _report attached
        filename = base_name + '_report.html'

        try:
            html_file = open(filename, 'w')
            html_file.write("<html>\n<head>\n<title>Leveling Report</title>\n")
            html_file.write("<style>table {border-collapse: collapse; width: 100%;} th, td {border: 1px solid black; padding: 8px; text-align: center;} body {font-family: Arial, sans-serif; margin: 40px;}</style>\n")
            html_file.write("</head>\n<body>\n")

            html_file.write("<h2>Leveling Computation Report</h2>\n")
            html_file.write(self.df.to_html(index=False, classes='table'))

            html_file.write("<h2>Misclosure and Accuracy Summary</h2>\n")
            html_file.write(f"<p><b>Error of Misclosure:</b> {self.error}</p>\n")
            html_file.write(f"<p><b>Order of Accuracy:</b> {self.order}</p>\n")

            html_file.write("</body>\n</html>\n")
            html_file.close()

            print('\nOkay, report saved as', filename)

            # Saving Graph
            self.save_graph(base_name)

        except Exception as e:
            print('Oops, error saving html file:', e)

    def run_menu(self):
        print("\n--- EXPORT MENU ---")
        print("Choose your report format (A profile graph will automatically be generated):")
        print("1 - Text file (.txt)")
        print("2 - HTML file (.html)")

        is_valid_choice = False
        while not is_valid_choice:
            user_choice = input("Enter 1 or 2: ")

            if user_choice == '1':
                self.save_to_txt()
                is_valid_choice = True
            elif user_choice == '2':
                self.save_to_html()
                is_valid_choice = True
            else:
                print("Invalid choice. Try again.")

In [107]:
# Calculate misclosure and accuracy
my_misclosure, my_accuracy = solver.misclosure()

# Calculate the adjusted elevations based on the misclosure
solver.adjusted_elev(my_misclosure)

# Create the report object and pass in the updated dataframe (data_2) and our calculated results
my_report = MakeReport(data_2, my_misclosure, my_accuracy)

# Run the export menu to let the user choose txt or html
my_report.run_menu()


--- EXPORT MENU ---
Choose your report format (A profile graph will automatically be generated):
1 - Text file (.txt)
2 - HTML file (.html)
Enter 1 or 2: 1
Type the base project name (e.g. Site_A): Site_B

Okay, report saved as Site_B_report.txt
Graph automatically saved as Site_B_profile.png
